# Notebook 04 — Unified ML: Train in Databricks MLflow, Score in Fabric

**Role:** Data Scientist

**What this shows:** A fraud detection model trained with MLflow (Databricks-style), registered in a model registry, then loaded and scored in Fabric — demonstrating the end-to-end ML lifecycle spanning both platforms.

> 🗣️ **Talking Point:** Data scientists stay in Databricks using MLflow, AutoML, or custom frameworks. Fabric ML uses the same MLflow API. Models are portable — train anywhere, serve anywhere. No model rewrite, no format conversion.

In [ ]:
# Import scikit-learn, MLflow, and PySpark functions for the unified ML pipeline
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from pyspark.sql import functions as F

# Confirm MLflow is available — same API used in Databricks and Fabric
print(f'MLflow version: {mlflow.__version__}')
print('Starting unified ML pipeline...')

## Step 1 — Prepare Features from Databricks-Sourced Data

> 🗣️ **Talking Point:** The feature data comes from the Databricks pipeline. The data scientist does not need to re-ingest or re-transform — they work directly on the Gold table that engineering produced.

In [ ]:
# Load the Gold transactions table produced by the Databricks ingestion pipeline
gold_df = spark.table('fabric_gold_transactions').toPandas()

# Encode the RiskTier label as a binary fraud indicator for the classifier target
# High-risk transactions are treated as positive fraud signals
gold_df['IsFraud'] = (gold_df['RiskTier'] == 'High').astype(int)

# Select numeric and encoded features available from the Databricks pipeline output
gold_df['IsLargeTransaction'] = gold_df['IsLargeTransaction'].astype(int)
gold_df['AmountBand_enc'] = pd.Categorical(gold_df['AmountBand']).codes
gold_df['Channel_enc']    = pd.Categorical(gold_df['Channel']).codes
gold_df['Region_enc']     = pd.Categorical(gold_df['Region']).codes

# Define the feature set for the fraud detection model
feature_cols = ['Amount','IsLargeTransaction','AmountBand_enc','Channel_enc','Region_enc']
X = gold_df[feature_cols].fillna(0)
y = gold_df['IsFraud']

# Split into training and test sets with stratification to handle class imbalance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f'Train: {len(X_train)} | Test: {len(X_test)} | Fraud rate: {y.mean():.1%}')

## Step 2 — Train with MLflow Tracking (Databricks-Compatible)

> 🗣️ **Talking Point:** This is exactly how a Databricks data scientist would train the model. Same MLflow `start_run()` / `log_metric()` / `log_model()` API. The experiment is portable — you can move it between Databricks and Fabric without code changes.

In [ ]:
# Train a Gradient Boosting fraud detection model with full MLflow experiment tracking
# MLflow API is identical in Databricks and Fabric — experiment is portable between platforms
mlflow.set_experiment('fabric_databricks_fraud_detection')

with mlflow.start_run(run_name='gbm_fraud_v1') as run:
    # Log hyperparameters so the experiment is fully reproducible
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('learning_rate', 0.1)
    mlflow.log_param('max_depth', 3)
    mlflow.log_param('data_source', 'databricks_pipeline_gold')

    # Train the Gradient Boosting Classifier on the Databricks-sourced features
    model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate and log AUC and accuracy to the MLflow experiment run
    y_prob = model.predict_proba(X_test)[:, 1]
    auc    = roc_auc_score(y_test, y_prob) if len(y_test.unique()) > 1 else 0.5
    acc    = model.score(X_test, y_test)
    mlflow.log_metric('roc_auc',  round(auc, 4))
    mlflow.log_metric('accuracy', round(acc, 4))

    # Register the model in MLflow Model Registry — same registry accessible from Databricks
    mlflow.sklearn.log_model(model, 'fraud_model', registered_model_name='FraudDetectionModel')

    run_id = run.info.run_id

# Print the training results and MLflow run details
print(f'MLflow Run ID : {run_id}')
print(f'ROC-AUC       : {auc:.4f}')
print(f'Accuracy      : {acc:.4f}')
print()
print('Model registered: FraudDetectionModel')
print('Same model registry is accessible from Databricks workspace via MLflow API.')

## Step 3 — Score New Transactions in Fabric

> 🗣️ **Talking Point:** The model was trained in Databricks (MLflow), but scoring runs in Fabric. Business users can trigger scoring via Power Automate or schedule it as a Fabric pipeline — no Databricks cluster spinning up for inference.

In [ ]:
# Load the registered MLflow model and score all transactions in Fabric
loaded_model = mlflow.sklearn.load_model(f'runs:/{run_id}/fraud_model')

# Apply the model to all transactions to generate fraud probability scores
gold_df['FraudProbability'] = loaded_model.predict_proba(gold_df[feature_cols].fillna(0))[:, 1]
gold_df['FraudFlag']        = (gold_df['FraudProbability'] >= 0.5).astype(int)

# Write the scored results back to a Delta table for Power BI consumption
scored_spark = spark.createDataFrame(
    gold_df[['TransactionID','AccountID','CustomerID','Amount','Region',
             'Channel','RiskTier','FraudProbability','FraudFlag']]
)
scored_spark.write.format('delta').mode('overwrite').saveAsTable('gold_fraud_scores')

# Print the fraud scoring summary for the demo audience
total      = len(gold_df)
flagged    = gold_df['FraudFlag'].sum()
high_risk  = (gold_df['FraudProbability'] >= 0.7).sum()
print(f'Scored {total} transactions')
print(f'Fraud-flagged : {flagged} ({flagged/total:.1%})')
print(f'High-risk (p>=0.7): {high_risk}')
print()
print('gold_fraud_scores written — ready for Power BI fraud monitoring dashboard.')

## Step 4 — Unified ML Platform Summary

> 🗣️ **Talking Point:** This is the complete story. Databricks owns data engineering and model training. Fabric owns serving, monitoring, and business consumption. The seam between them is invisible to end users.

In [ ]:
# Print the end-to-end platform responsibility summary for the demo audience
summary = [
    ('Databricks',  'Raw ingestion, Delta Live Tables, heavy ETL, MLflow training, Unity Catalog'),
    ('Delta Share', 'Zero-copy hand-off via OneLake shortcut — no ETL, no duplication'),
    ('Fabric',      'Semantic model, Power BI Direct Lake, ML scoring, business user access'),
    ('Governance',  'Microsoft Purview spans both platforms — unified audit and lineage'),
    ('Cost',        'Storage in ADLS Gen2 once; compute split by workload type'),
]

# Display the platform responsibility matrix
print('=== Better Together Platform Summary ===')
for platform, responsibility in summary:
    print(f'  {platform:12}: {responsibility}')